# Lab: Convolutional Neural Networks for Satellite Image Classification

**Instructor:** Muhammad Sayed  
**Semester:** Spring 2026

---

### Learning Objectives
* A custom **Convolutional Neural Network** architecture is designed and implemented using PyTorch.
* **High-resolution satellite imagery** is mapped from fine-grained categories to broad macro-classes.
* **Classification model generalization** is evaluated through systematic **hyperparameter tuning**.
* Classification errors are analyzed by **tracing misclassified macro-classes back to their original micro-class spectral signatures**.
* Results and experimental trials are serialized utilizing Pydantic models for **standardized** evaluation.


In [109]:
import os
import json
from enum import Enum
from typing import List, Dict, Any
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset,random_split,Dataset
from torchvision.datasets import ImageFolder
from torchvision import transforms
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from pydantic import BaseModel
import warnings
torch.manual_seed(42)

warnings.filterwarnings('ignore')

In [96]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("aqibrehmanpirzada/nwpuresisc45")

print("Path to dataset files:", path)


Using Colab cache for faster access to the 'nwpuresisc45' dataset.
Path to dataset files: /kaggle/input/nwpuresisc45


In [97]:
!cp -r  /root/.cache/kagglehub/datasets/aqibrehmanpirzada/nwpuresisc45/versions/1 /content/Dataset

### Data Engineering and Macro-Class Mapping
The required NWPU-RESISC45 dataset can be downloaded from [Kaggle](https://www.kaggle.com/datasets/aqibrehmanpirzada/nwpuresisc45). It consists of forty-five fine-grained classes. These are algorithmically mapped to four broad macro-classes. Cloud cover is categorized as unknown and must be filtered out during dataset initialization.

In [103]:
class LandCover(Enum):
    UNKNOWN = 0
    GREENERY = 1
    SAND = 2
    WATER = 3
    CEMENT_INFRASTRUCTURE = 4

macro_class_mapping = {
    'chaparral': LandCover.GREENERY,
    'circular_farmland': LandCover.GREENERY,
    'forest': LandCover.GREENERY,
    'golf_course': LandCover.GREENERY,
    'meadow': LandCover.GREENERY,
    'rectangular_farmland': LandCover.GREENERY,
    'terrace': LandCover.GREENERY,

    'beach': LandCover.SAND,
    'desert': LandCover.SAND,
    'mountain': LandCover.SAND,
    'island': LandCover.SAND,

    'harbor': LandCover.WATER,
    'lake': LandCover.WATER,
    'river': LandCover.WATER,
    'sea_ice': LandCover.WATER,
    'wetland': LandCover.WATER,
    'ship': LandCover.WATER,
    'snowberg': LandCover.WATER,

    'airport': LandCover.CEMENT_INFRASTRUCTURE,
    'bridge': LandCover.CEMENT_INFRASTRUCTURE,
    'church': LandCover.CEMENT_INFRASTRUCTURE,
    'commercial_area': LandCover.CEMENT_INFRASTRUCTURE,
    'dense_residential': LandCover.CEMENT_INFRASTRUCTURE,
    'freeway': LandCover.CEMENT_INFRASTRUCTURE,
    'industrial_area': LandCover.CEMENT_INFRASTRUCTURE,
    'intersection': LandCover.CEMENT_INFRASTRUCTURE,
    'medium_residential': LandCover.CEMENT_INFRASTRUCTURE,
    'mobile_home_park': LandCover.CEMENT_INFRASTRUCTURE,
    'overpass': LandCover.CEMENT_INFRASTRUCTURE,
    'palace': LandCover.CEMENT_INFRASTRUCTURE,
    'parking_lot': LandCover.CEMENT_INFRASTRUCTURE,
    'railway': LandCover.CEMENT_INFRASTRUCTURE,
    'railway_station': LandCover.CEMENT_INFRASTRUCTURE,
    'roundabout': LandCover.CEMENT_INFRASTRUCTURE,
    'runway': LandCover.CEMENT_INFRASTRUCTURE,
    'sparse_residential': LandCover.CEMENT_INFRASTRUCTURE,
    'storage_tank': LandCover.CEMENT_INFRASTRUCTURE,
    'thermal_power_station': LandCover.CEMENT_INFRASTRUCTURE,
    'baseball_diamond': LandCover.CEMENT_INFRASTRUCTURE,
    'basketball_court': LandCover.CEMENT_INFRASTRUCTURE,
    'ground_track_field': LandCover.CEMENT_INFRASTRUCTURE,
    'stadium': LandCover.CEMENT_INFRASTRUCTURE,
    'tennis_court': LandCover.CEMENT_INFRASTRUCTURE,
    'airplane': LandCover.CEMENT_INFRASTRUCTURE,

    'cloud': LandCover.UNKNOWN
}

### Dataset Initialization and Loading
The provided dataset directory is natively partitioned into training and testing subsets. Images must be loaded and mapped to their respective macro-classes, ensuring instances mapped to the `UNKNOWN` category are explicitly excluded. The testing subset is to be preserved strictly for final evaluation. To facilitate hyperparameter tuning without data leakage, the training subset must be further partitioned to generate an independent validation subset, or a cross-validation strategy must be implemented.

In [104]:
# A transform pipeline is required (e.g., resizing, tensor conversion, normalization).
class loadDataset(Dataset):
    def __init__(self, subset, labels):
        self.subset = subset
        self.labels = labels

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        image, _ = self.subset[idx]
        label = self.labels[idx]
        return image, label

def mapping_func(dataset):
    valid_idx=[]
    mapping=[]
    for idx,(_,label)in enumerate(dataset.samples):
        name=dataset.classes[label]
        class_name=macro_class_mapping[name]
        if class_name!=LandCover.UNKNOWN:
            valid_idx.append(idx)
            mapping.append(class_name.value-1)
    return valid_idx,mapping

transformer=transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = ImageFolder(root='/content/Dataset/Dataset/train/train', transform=transformer)
test_dataset = ImageFolder(root='/content/Dataset/Dataset/test/test', transform=transformer)

train_valid_idx,train_map=mapping_func(train_dataset)
test_valid_idx,test_map=mapping_func(test_dataset)

train_valid_dataset=Subset(train_dataset,train_valid_idx)
test_valid_dataset=Subset(test_dataset,test_valid_idx)

train_size=int(0.8*len(train_valid_dataset))
valid_size=len(train_valid_dataset)-train_size

train_data, valid_data = random_split(train_valid_dataset, [train_size, valid_size])

print("Training dataset size:", len(train_data))
print("Validation dataset size:", len(valid_data))
print("Testing dataset size:", len(test_valid_dataset))

train_labels=[train_map[idx] for idx in train_data.indices]
valid_labels=[train_map[idx] for idx in valid_data.indices]

print("training labels size:", len(train_labels))
print("validation labels size:", len(valid_labels))

train=loadDataset(train_data, train_labels)
valid=loadDataset(valid_data, valid_labels)
test=loadDataset(test_valid_dataset, test_map)

batch_size=32
train_loader=DataLoader(train, batch_size=batch_size, shuffle=True)
valid_loader=DataLoader(valid, batch_size=batch_size, shuffle=False)
test_loader=DataLoader(test, batch_size=batch_size, shuffle=False)

# Valid indices and mapped targets are to be extracted for both datasets, excluding the UNKNOWN class.
# The train_dataset must be dynamically split into distinct training and validation subsets.
# DataLoaders for the training, validation, and testing subsets are to be instantiated here.


Training dataset size: 21120
Validation dataset size: 5280
Testing dataset size: 4400
training labels size: 21120
validation labels size: 5280


### Convolutional Neural Network Architecture
A custom CNN class is defined. Spatial dimensions must be carefully tracked between convolutional outputs and fully connected layer inputs.

As a reference model, the LeNet-5 architecture illustrates the foundational principles of convolutional networks. The input tensor is sequentially processed through alternating convolutional and pooling layers to extract spatial features and reduce dimensionality. The resulting feature maps are subsequently flattened and passed through fully connected layers to yield the final classification probabilities.


In [105]:
class SatelliteCNN(nn.Module):
    def __init__(self):
        super(SatelliteCNN, self).__init__()

        self.conv1 = nn.Conv2d(3, 8, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(2, 2)
        self.conv3 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool3 = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(32*16*16, 128)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(128, 4)
    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = self.pool1(x)
        x = torch.relu(self.conv2(x))
        x = self.pool2(x)
        x = torch.relu(self.conv3(x))
        x = self.pool3(x)
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x


### Model Training
The loss function and optimizer are initialized. The training loop is executed, updating model weights while validating performance on the held-out validation set to monitor for overfitting.

In [106]:
# Model, Loss Criterion, and Optimizer are instantiated here.
# The training loop over defined epochs is executed here.
model=SatelliteCNN()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters(), lr=0.001)

epochs=10
def training_model(model,epochs,criterion,optimizer):
  for epoch in range(epochs):
      model.train()
      training_loss=0.0
      for images, labels in train_loader:
          images, labels = images.to(device), labels.to(device)
          optimizer.zero_grad()
          outputs = model(images)
          loss = criterion(outputs, labels)
          loss.backward()
          optimizer.step()
          training_loss += loss.item() * images.size(0)
      epoch_loss = training_loss / len(train_loader.dataset)
      print(f'Epoch {epoch+1}/{epochs}, Training Loss: {epoch_loss:.4f}')

      model.eval()
      valid_loss=0.0
      with torch.no_grad():
          for images, labels in valid_loader:
              images, labels = images.to(device), labels.to(device)
              outputs = model(images)
              loss = criterion(outputs, labels)
              valid_loss += loss.item() * images.size(0)
      epoch_valid_loss = valid_loss / len(valid_loader.dataset)
      print(f'Epoch {epoch+1}/{epochs}, Validation Loss: {epoch_valid_loss:.4f}')
training_model(model,epochs,criterion,optimizer)

Epoch 1/10, Training Loss: 0.8087
Epoch 1/10, Validation Loss: 0.6619
Epoch 2/10, Training Loss: 0.6436
Epoch 2/10, Validation Loss: 0.5431
Epoch 3/10, Training Loss: 0.5705
Epoch 3/10, Validation Loss: 0.5344
Epoch 4/10, Training Loss: 0.5103
Epoch 4/10, Validation Loss: 0.4621
Epoch 5/10, Training Loss: 0.4545
Epoch 5/10, Validation Loss: 0.4638
Epoch 6/10, Training Loss: 0.4065
Epoch 6/10, Validation Loss: 0.4566
Epoch 7/10, Training Loss: 0.3637
Epoch 7/10, Validation Loss: 0.4654
Epoch 8/10, Training Loss: 0.3191
Epoch 8/10, Validation Loss: 0.4688
Epoch 9/10, Training Loss: 0.2895
Epoch 9/10, Validation Loss: 0.4658
Epoch 10/10, Training Loss: 0.2616
Epoch 10/10, Validation Loss: 0.4812


### Quantitative Evaluation
The trained model is evaluated against the test subset. Overall Accuracy, Macro-Averaged F1 Scores, and the Confusion Matrix are computed.

In [107]:
# Predictions on the test set are generated.
# accuracy_score, f1_score, and confusion_matrix are computed and printed.
def evaluate_model(model):
  model.eval()
  test_loss = 0.0
  test_preds = []
  test_labels = []
  with torch.no_grad():
      for images, labels in test_loader:
          images, labels = images.to(device), labels.to(device)
          outputs = model(images)
          _,preds=torch.max(outputs,1)
          test_preds.extend(preds.cpu().numpy())
          test_labels.extend(labels.cpu().numpy())
  acc=accuracy_score(test_labels, test_preds)
  print(f'Test Accuracy: {acc:.2f}')
  f1=f1_score(test_labels, test_preds,average=None)
  for i,f1 in enumerate(f1,1):
    print(f'{LandCover(i).name}: {f1:.2f}')
  cm=confusion_matrix(test_labels,test_preds)
  print("confusion matrix",cm)
evaluate_model(model)

Test Accuracy: 0.85
GREENERY: 0.78
SAND: 0.75
WATER: 0.72
CEMENT_INFRASTRUCTURE: 0.92
confusion matrix [[ 535   30   49   86]
 [  27  296   44   33]
 [  49   29  488  134]
 [  61   31   77 2431]]


### Hyperparameter Tuning Analysis
Observations regarding hyperparameter tuning are required. Multiple trials must be recorded, documenting the chosen parameters, resultant accuracy, class-specific F-scores, and drawn conclusions. These records are to be serialized using the provided Pydantic schema in the submission cell.


In [94]:
# Hyperparameter tuning execution is to be implemented here.
# Utilizing a grid search strategy (e.g., via itertools.product or sklearn.model_selection.ParameterGrid) is suggested to systematically explore combinations of learning rates, batch sizes, optimizers, and epochs.
import itertools

learning_rates = [0.001, 0.0005]
batch_sizes = [16, 32]
optimizers = ['Adam', 'SGD']
epochs_list = [5,10]

param_grid = list(itertools.product(learning_rates, batch_sizes, optimizers, epochs_list))
print("Total trials:", len(param_grid))
for lr, batch_size, opt, epochs in param_grid:
    print(f"Trial: lr={lr}, batch_size={batch_size}, optimizer={opt}, epochs={epochs}")
    model=SatelliteCNN()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    criterion=nn.CrossEntropyLoss()
    train_loader=DataLoader(train, batch_size=batch_size, shuffle=True)
    valid_loader=DataLoader(valid, batch_size=batch_size, shuffle=False)
    test_loader=DataLoader(test, batch_size=batch_size, shuffle=False)

    if opt == 'Adam':
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    elif opt == 'SGD':
        optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    training_model(model,epochs,criterion,optimizer)
    evaluate_model(model)


Total trials: 16
Trial: lr=0.001, batch_size=16, optimizer=Adam, epochs=5
Epoch 1/5, Training Loss: 0.7988
Epoch 1/5, Validation Loss: 0.6256
Epoch 2/5, Training Loss: 0.6149
Epoch 2/5, Validation Loss: 0.5387
Epoch 3/5, Training Loss: 0.5406
Epoch 3/5, Validation Loss: 0.5477
Epoch 4/5, Training Loss: 0.4840
Epoch 4/5, Validation Loss: 0.4693
Epoch 5/5, Training Loss: 0.4307
Epoch 5/5, Validation Loss: 0.4819
Test Accuracy: 0.82
GREENERY: 0.75
SAND: 0.69
WATER: 0.66
CEMENT_INFRASTRUCTURE: 0.90
confusion matrix [[ 573   29   32   66]
 [  63  286   19   32]
 [  72   59  402  167]
 [ 110   55   70 2365]]
Trial: lr=0.001, batch_size=16, optimizer=Adam, epochs=10
Epoch 1/10, Training Loss: 0.8076
Epoch 1/10, Validation Loss: 0.6048
Epoch 2/10, Training Loss: 0.6484
Epoch 2/10, Validation Loss: 0.5597
Epoch 3/10, Training Loss: 0.5697
Epoch 3/10, Validation Loss: 0.5724
Epoch 4/10, Training Loss: 0.5112
Epoch 4/10, Validation Loss: 0.5095
Epoch 5/10, Training Loss: 0.4543
Epoch 5/10, Valida

### Error Analysis and Confusion Patterns
The macro-class exhibiting the lowest accuracy must be identified based on the computed F-scores and Confusion Matrix. Subsequently, the original, fine-grained micro-classes constituting the misclassified instances within this macro-class are to be examined. Patterns explaining why the convolutional neural network struggles to differentiate these specific land cover categories must be formulated and reported in the submission payload.


In [111]:
model.eval()
test_preds = []
test_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        preds = torch.argmax(outputs, dim=1)

        test_preds.extend(preds.cpu().numpy())
        test_labels.extend(labels.numpy())

f1_scores = f1_score(test_labels, test_preds, average=None)
macro_min = np.argmin(f1_scores)

print("Least macro class:", LandCover(macro_min+1).name)

test_micro = [
    test_dataset.classes[label]
    for _, label in [test_dataset.samples[i] for i in test_valid_idx]
]

filtered_micro = []
filtered_preds = []

for true, pred, micro in zip(test_labels, test_preds, test_micro):
    if true == macro_min:
        filtered_micro.append(micro)
        filtered_preds.append(pred)
df = pd.DataFrame({
    "micro class": filtered_micro,
    "predicted macro": filtered_preds
})
crosstab = pd.crosstab(df["micro class"], df["predicted macro"])

print(crosstab)


Least macro class: WATER
predicted macro   0  1   2   3
micro class                   
harbor            0  2  76  22
lake              4  3  85   8
river            12  7  68  13
sea_ice           0  2  97   1
ship              3  8  46  43
snowberg          0  0  69  31
wetland          30  7  47  16


### Pydantic Models and Submission Export
The `pydantic` library is utilized to ensure structural integrity of the final submission. The models below define the exact schema expected for automated grading. The variables are to be populated with the experimental findings, and the cell executed to write the JSON output.

In [ ]:
class TrialRecord(BaseModel):
    hyperparams: Dict[str, Any]
    accuracy: float
    F_score_class: Dict[str, float]
    observations: List[str]
    conclusions: List[str]

class LabSubmission(BaseModel):
    student_ids: List[str]
    trials: List[TrialRecord]
    micro_class_error_analysis: str

# The records are populated here based on experimental iterations.
trial_one = TrialRecord(
    hyperparams={"learning_rate": 0.001, "batch_size": 16, "optimizer": "Adam", "epochs": 5},
    accuracy= 0.82,
    F_score_class={"GREENERY":0.75, "SAND": 0.69, "WATER": 0.66, "CEMENT_INFRASTRUCTURE": 0.90},
    observations=[ "Validation loss increased while training loss decreased."],
    conclusions=["A lower learning rate may be required.", "Further regularization is recommended.","Data Augmentation"]
)


trail_two = TrialRecord(
    hyperparams={"learning_rate": 0.001, "batch_size": 16, "optimizer": "Adam", "epochs": 10},
    accuracy=0.84,
    F_score_class={"GREENERY":0.75, "SAND": 0.74, "WATER": 0.72, "CEMENT_INFRASTRUCTURE": 0.91},
    observations=[ "Validation loss fluctuated"],
    conclusions=["A lower learning rate may be required.", "Further regularization is recommended."]
)

trail_three = TrialRecord(
    hyperparams={"learning_rate": 0.001, "batch_size": 16, "optimizer": "SGD", "epochs": 5},
    accuracy=0.79,
    F_score_class={"GREENERY":0.68, "SAND": 0.64, "WATER": 0.59, "CEMENT_INFRASTRUCTURE": 0.88},
    observations=[ "Training and Validation error decreased but f1 scores decreased also"],
    conclusions=["increase epochs"]
)
trail_four = TrialRecord(
    hyperparams={"learning_rate": 0.001, "batch_size": 16, "optimizer": "SGD", "epochs":10},
    accuracy=0.79,
    F_score_class={"GREENERY":0.71, "SAND": 0.69, "WATER": 0.58, "CEMENT_INFRASTRUCTURE": 0.88},
    observations=[ "Validation loss fluctuated"],
    conclusions=["A lower learning rate may be required.", "Further regularization is recommended."]
)

trial_five = TrialRecord(
    hyperparams={"learning_rate": 0.001, "batch_size": 32, "optimizer": "Adam", "epochs": 5},
    accuracy= 0.80,
    F_score_class={"GREENERY":0.72, "SAND": 0.65, "WATER": 0.51, "CEMENT_INFRASTRUCTURE": 0.89},
    observations=[ "Validation loss increased while training loss decreased.","less than when batch size 16"],
    conclusions=["A lower learning rate may be required.", "Further regularization is recommended.","Data Augmentation"]
)

trail_six = TrialRecord(
    hyperparams={"learning_rate": 0.001, "batch_size": 32, "optimizer": "Adam", "epochs": 10},
    accuracy=0.83,
    F_score_class={"GREENERY":0.75, "SAND": 0.72, "WATER": 0.65, "CEMENT_INFRASTRUCTURE": 0.91},
    observations=[ "Validation loss fluctuated"],
    conclusions=["A lower learning rate may be required.", "Further regularization is recommended."]
)

trail_seven = TrialRecord(
    hyperparams={"learning_rate": 0.001, "batch_size": 32, "optimizer": "SGD", "epochs": 5},
    accuracy=0.76,
    F_score_class={"GREENERY":0.65, "SAND": 0.64, "WATER": 0.52, "CEMENT_INFRASTRUCTURE": 0.86},
    observations=[ "Training and Validation error decreased but f1 scores decreased also"],
    conclusions=["increase epochs"]
)

trail_eight = TrialRecord(
    hyperparams={"learning_rate": 0.001, "batch_size": 32, "optimizer": "SGD", "epochs":10},
    accuracy=0.79,
    F_score_class={"GREENERY":0.65, "SAND": 0.65, "WATER": 0.56, "CEMENT_INFRASTRUCTURE": 0.88},
    observations=[ "Validation loss fluctuated"],
    conclusions=["A lower learning rate may be required.", "Further regularization is recommended."]
)

trial_nine = TrialRecord(
    hyperparams={"learning_rate": 0.0005, "batch_size": 16, "optimizer": "Adam", "epochs": 5},
    accuracy= 0.81,
    F_score_class={"GREENERY":0.73, "SAND": 0.71, "WATER": 0.56, "CEMENT_INFRASTRUCTURE": 0.89},
    observations=[ "Validation loss increased while training loss decreased."],
    conclusions=["A lower learning rate may be required.", "Further regularization is recommended.","Data Augmentation"]
)

trail_ten = TrialRecord(
    hyperparams={"learning_rate": 0.0005, "batch_size": 16, "optimizer": "Adam", "epochs": 10},
    accuracy=0.83,
    F_score_class={"GREENERY":0.78, "SAND": 0.77, "WATER": 0.60, "CEMENT_INFRASTRUCTURE": 0.90},
    observations=[ "Validation loss fluctuated"],
    conclusions=["A lower learning rate may be required.", "Further regularization is recommended."]
)

trail_11 = TrialRecord(
    hyperparams={"learning_rate": 0.0005, "batch_size": 16, "optimizer": "SGD", "epochs": 5},
    accuracy=0.74,
    F_score_class={"GREENERY":0.55, "SAND": 0.64, "WATER": 0.56, "CEMENT_INFRASTRUCTURE": 0.85},
    observations=[ "Training and Validation error decreased but f1 scores decreased also"],
    conclusions=["increase epochs"]
)

trail_12 = TrialRecord(
    hyperparams={"learning_rate": 0.0005, "batch_size": 16, "optimizer": "SGD", "epochs":10},
    accuracy=0.80,
    F_score_class={"GREENERY":0.70, "SAND": 0.67, "WATER": 0.56, "CEMENT_INFRASTRUCTURE": 0.89},
    observations=[ "Training and Validation error decreased"],
    conclusions=["may increase epoch improve accuracy and f1"]
)

trial_13 = TrialRecord(
    hyperparams={"learning_rate": 0.0005, "batch_size": 32, "optimizer": "Adam", "epochs": 5},
    accuracy= 0.82,
    F_score_class={"GREENERY":0.70, "SAND": 0.70, "WATER": 0.76, "CEMENT_INFRASTRUCTURE": 0.89},
    observations=[ "Training and Validation error decreased","better then when 16 batch size"],
    conclusions=["may increase epoch improve accuracy and f1"]
)

trail_14 = TrialRecord(
    hyperparams={"learning_rate": 0.0005, "batch_size": 32, "optimizer": "Adam", "epochs": 10},
    accuracy=0.83,
    F_score_class={"GREENERY":0.75, "SAND": 0.74, "WATER": 0.69, "CEMENT_INFRASTRUCTURE": 0.90},
    observations=[ "Validation loss fluctuated"],
    conclusions=["A lower learning rate may be required.", "Further regularization is recommended."]
)

trail_15 = TrialRecord(
    hyperparams={"learning_rate": 0.0005, "batch_size": 32, "optimizer": "SGD", "epochs": 5},
    accuracy=0.69,
    F_score_class={"GREENERY":0.53, "SAND": 0.60, "WATER": 0.07, "CEMENT_INFRASTRUCTURE": 0.82},
    observations=[ "training loss and validation loss is very hight","water class is very bad"],
    conclusions=["increase epochs"]
)


trail_16 = TrialRecord(
    hyperparams={"learning_rate": 0.0005, "batch_size": 32, "optimizer": "SGD", "epochs":10},
    accuracy=0.78,
    F_score_class={"GREENERY":0.67, "SAND": 0.65, "WATER": 0.53, "CEMENT_INFRASTRUCTURE": 0.87},
    observations=[ "training loss and validation loss decreased"],
    conclusions=["increase epochs"]
)
# Additional trials are to be added to the list below.
submission = LabSubmission(
    student_ids=[9220912, 9220658],
    trials=[trial_one,trail_two,trail_three,trail_four,trial_five,trail_six,trail_seven,trail_eight,trial_nine,trail_ten],
    micro_class_error_analysis="Insert the detailed analysis required by Question D here, discussing the specific original micro-classes that were misclassified within the worst-performing macro-class."
)

# The Pydantic model is serialized to JSON and saved for grading.
output_json = submission.model_dump_json(indent=4)
print(output_json)

with open("student_submission.json", "w") as f:
    f.write(output_json)


### Submission Protocol

The final deliverable **must be a zipped archive** submitted via Google Classroom.

* The archive must contain the executed Jupyter Notebook and the generated `student_submission.json` file.
* The zipped file must be explicitly named utilizing the display name of the submitter exactly as it appears on Google Classroom.
* **Failure to adhere to these archival and naming conventions disrupts automated grading pipelines and will result in direct grade penalties.**

---

### Grading Rubric (Total: Ten Marks)

The requirement submission is evaluated based on the following criteria:

* **Data Engineering and Macro-Class Mapping (Two Marks)**
  The dataset is correctly loaded, successfully mapped to the defined Enum macro-classes, and securely partitioned into distinct training, validation, and testing subsets without data leakage.

* **Architecture and Training Execution (Two Marks)**
  A functional Convolutional Neural Network is constructed with appropriate spatial dimension tracking, and the training loop successfully computes gradients and updates model weights.

* **Hyperparameter Tuning (Two Marks)**
  Systematic hyperparameter tuning is executed, and multiple experimental trials are accurately serialized within the final JSON payload.

* **Error Analysis and Custom Confusion Matrix (Two Marks)**
  A custom confusion matrix mapping true micro-classes against predicted macro-classes is successfully generated, and systematic misclassification patterns for the least accurate class are logically analyzed in the written submission.

* **Quantitative Evaluation and Peer Comparison (Two Marks)**
  The macro-averaged F-score from the final experimental trial in the submitted list is explicitly reported and analytically compared against peer performances to identify relative architectural strengths and areas for optimization.
